# 04 · 公式SASRec

公式モデルとサンプラーはvendorに原文を保存しています。100epoch・batch256・64次元・2層2head・Adam0.001・dropout0・seed42を保ち、MPS（使えなければCPU）で29日／149日モデルを順に学習します。系列は公式同様、ユーザーごとの最後の200件です。

MPS非対応のnested tensor推論だけをwrapperで無効化します。2件未満の学習系列は次曲ペアが作れないため除外します。予測には起点以前の既知曲と条件を満たす今回の推薦だけを使います。

In [1]:
import polars as pl
from IPython.display import display
from relisten import config
from relisten.sequence_model import build_sasrec_features,device_name
from relisten.evaluation import evaluate_features
print("device:",device_name())

device: mps


In [2]:
features = build_sasrec_features()
display(features.select("model_end_day").group_by("model_end_day").len())

SASRec day29 epoch1/100: 4.1s loss=0.68610


SASRec day29 epoch2/100: 3.1s loss=0.61108


SASRec day29 epoch3/100: 3.2s loss=0.51352


SASRec day29 epoch4/100: 3.2s loss=0.44864


SASRec day29 epoch5/100: 3.1s loss=0.40849


...

SASRec day149 epoch97/100: 4.7s loss=0.02811


SASRec day149 epoch98/100: 4.7s loss=0.02762


SASRec day149 epoch99/100: 4.7s loss=0.02727


SASRec day149 epoch100/100: 4.7s loss=0.02687


model_end_day,len
i32,u32
29,1285567
149,3014337


In [3]:
metrics = evaluate_features(features,{"SASRec":"sasrec"},"sasrec_model_metrics.csv")
display(metrics.filter((pl.col("route")=="C") & (pl.col("split")=="test") & (pl.col("lower_days")==config.LOWER_BOUND_DAYS)))

evaluated L=1 T2 C


evaluated L=1 T2 B


evaluated L=1 T1 C


evaluated L=1 T1 B


evaluated L=3 T2 C


...

evaluated L=14 T2 C


evaluated L=14 T2 B


evaluated L=14 T1 C


evaluated L=14 T1 B


model,protocol,task,route,split,lower_days,is_main_lower,pairs,users,auc_users,positive_rate,user_auc,user_auc_ci_low,user_auc_ci_high,global_auc,pr_auc,log_loss,brier
str,str,str,str,str,i64,bool,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""UserBaseline""","""baseline""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.5,0.5,0.5,0.607669,0.135043,0.31237,0.086232
"""SASRec""","""score_only""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.531674,0.52688,0.536589,0.519614,0.106193,0.317167,0.087065
"""SASRec""","""baseline_plus_score""","""T2""","""C""","""test""",1,true,967884,4819,4124,0.096279,0.531674,0.52688,0.536589,0.606905,0.13495,0.312525,0.086261
"""UserBaseline""","""baseline""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.5,0.5,0.5,0.614387,0.263604,0.484497,0.154408
"""SASRec""","""score_only""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.455199,0.450605,0.45961,0.483107,0.188726,0.490925,0.155868
"""SASRec""","""baseline_plus_score""","""T1""","""C""","""test""",1,true,982716,4485,3953,0.192999,0.455199,0.450605,0.45961,0.613284,0.262811,0.484592,0.154438
